In [ ]:
# config.py 

from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, ValidationError, field_validator

# 1️⃣ info.data
# A dictionary of other already-validated fields in the model.

# environment: str = Field(default="dev", description="dev|staging|prod")
# Means:
# It must be a string (str)
# If no value is provided → it automatically becomes "dev"
# The description is just metadata (mainly for docs, like in FastAPI)
# Imagine a thermostat.
# It must hold a number (temperature type requirement).
# If no one sets it, it defaults to 72°F.


# What does ./ mean?
# ./ means:
# “Start from the current directory.”
# Simple Analogy 🏠
# Imagine you’re standing in your house.
# ./ = “Right here, where I’m standing.”
# ./garage = “The garage inside this house.”
# ./.rag_store = “The .rag_store room inside this house.”
# It does not mean “go somewhere else.”
# It means “use a folder in the same place I’m currently in.”

#settings.py 
class Settings(BaseSettings):
    model_config = SettingsConfigDict()
    environment: str = Field(default="dev", description="dev|staging")
    app_name: str = ""
    host: str = "0.0.0.0"
    port: int = 800 
    log_level: str = "info"



    @field_validator("openai_api_key", mode="before")
    @classmethod
    def validate_openai_api_key(cls, v, info):
        env = info.data.get("environment", "dev")
        if env in ["prod", "production", "staging"] and (not v or v == "sk-..."):
            raise ValueError(
                "OPENAI_API_KEY must be set in production/staging environments."
            )
        return v


















# env_prefix="MRC_"
# Only listen to customization switches labeled for this car model.
# env_file=".env"
# Read special car customizations from the factory clipboard file.
# extra="ignore"
# If someone adds a feature this car doesn’t support, just ignore it.
# app_name: str = "Monster Resort Concierge"
# The car’s official name badge if nobody renames it.
# environment: str = Field(default="dev", description="dev|staging|prod")
# The car starts in garage mode ("dev") unless you switch it to test track or showroom mode.
# host: str = "0.0.0.0"
# The car is open to be accessed from anywhere.
# port: int = 8000
# The car parks in spot 8000 unless told otherwise.
# log_level: str = "info"
# The dashboard speaks at a normal, informative level by default.



# cls → the model class
# v → the value of openai_api_key
# info → metadata about the model during validation

# What is this line? # What is this line?
# env = info.data.get("environment", "dev")
# This means:
# “Look at the other fields already provided in this model.
# If there is a field called environment, use it.
# If not, default to "dev".”
# So if someone creates:
# Settings(
#     openai_api_key="abc",
#     environment="prod"
# )
# Then:
# env == "prod"
# If they don’t provide environment:
# env == "dev"

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import sqlite3
from typing import Iterator, Optional
from app.logging_utils import DatabaseError

#database.py

# What it was probably intended for: threading.local() gives each thread its own
#    private storage, typically used to give each thread its own SQLite connection
#    (since SQLite connections aren't safely shared across threads). The pattern  
#   would look like:

#   def get_connection(self):
#       if not hasattr(self._local, "conn"):
#           self._local.conn = sqlite3.connect(self.db_path)
#       return self._local.conn

#   But your get_connection() creates a new connection every call and closes it in
#    the finally block. That's a valid alternative approach — it just makes _local
#    pointless.

#   Is it acceptable? No — unused code is noise. It suggests an incomplete
#   implementation and would raise a question in a code review. You should either
#   remove it or use it. Since your open-and-close-per-call pattern already works
#   and is thread-safe (each call gets its own connection), just remove it.

#   Want me to remove it?

DDL = [
   """ CREATE TABLE IF NOT EXISTS akin (
   name INT 
   age INT 
   )""",

   """" CREATE TABLE IF NOT EXISTS animal (
    food TEXT 
    items TEXT
   )
   """
]


class DatabaseManager:
   def __init__(self, settings: Settings):
      self.db_path = Path(getattr(settings, "monster_db", "sqlite_db"))
      self.db_path.mkdir(parents=True, exist_ok=True)
      pass




class DatabaseManager:
   def __init__(self, settings: Settings):
      self.db_path = Path(getattr(settings, "mosterdb", "sqlutedb"))
      self.db_path.mkdir(parents=True, exist_ok=True)


   @contextmanager
   def get_connection(self) -> Iterator[sqlite3.Connection]:
      conn = sqlite3.connect(database=self.db_path, check_same_thread=True)
      conn.row_factory = sqlite3.Row 

      try:
         yield conn 
         conn.commit()
      except Exception as e:
         conn.rollback()
         raise DatabaseError()
      finally:
         conn.close()

   @contextmanager
   def session(self) -> Iterator[sqlite3.Connection]:
      with self.get_connection as conn:
         yield conn         


   def get_booking(self, booking_id: str) -> Optional[dict]:
      query = "SELECT * FROM bookings WHERE bookingid = ? OR sessiinid is ?"

      try:
         with self.get_connection() as conn:
            cur = conn.execute(query, (booking_id, booking_id))
            row = cur.fetchone()
            return dict[row] if row else None 
      except Exception as e:
         raise DatabaseError()   






   # “I will set something up, give it to you, and clean it up afterward — even if something goes wrong.”
   # Without a context manager, people forget to close things 😬
   @contextmanager
   def get_connection() -> Iterator[sqlite3.Connection]:
      conn = sqlite3.connect("")
      conn.close()

      try:
         yield conn 
         conn.commit()
      except Exception as e:
         conn.rollback()
         raise DatabaseError()
      finally:
         conn.close()   



class DatabaseManager:
   def __init__(self):
      self._init_db()

   def _init_db(self):
      try:
         with self.get_connection() as conn:
            print()
      except Exception as e:
         print(e)   


   @contextmanager 
   # “I will set something up, give it to you, and clean it up afterward — even if something goes wrong.”
   # Without a context manager, people forget to close things 😬
   def get_connection2(self) -> Iterator[sqlite3.Connection]:
      conn = sqlite3.connect(self.db_path, check_same_thread=False)
      conn.row_factory = sqlite3.Row

   @contextmanager
   def get_session(self) -> Iterator[sqlite3.Connection]:
      with self.get_connection2() as conn: 
         yield conn    



   def get_session(self) -> Iterator[sqlite3.Connection]:
      with self.get_conn() as conn:
         yield conn 


      # DEFAULT FOR PRACTICE
      @contextmanager
      def get_connection(self) -> Iterator[sqlite3.Connection]:
         conn = sqlite3.connect(self.db_path, check_same_thread=False)
         conn.row_factory = sqlite3.Row
         try:
               yield conn
               conn.commit()
         except Exception as e:
               conn.rollback()
               raise DatabaseError(f"Database transaction failed: {e}")
         finally:
               conn.close()


class DatabaseManager:
   def __init__(self, settings: Settings):
      self.db_path = Path(getattr(settings, "sqlite db", "monster db"))
      self.db_path.parent.mkdir(parents=True, exist_ok=True)


   def get_connection(self) -> Iterator[sqlite3.Connection]:
      conn = sqlite3.connect(database="")
      conn.row_factory = sqlite3.Row 

      try:
         yield conn
         conn.commit
      except Exception as e:
         conn.rollback()
         raise DatabaseError()
      finally:
         conn.close()
     
        

# @contextmanager
# def session(self) -> Iterator[sqlite3.Connection]:
#     with self.get_connection() as conn:
#         yield conn
# What’s happening here
# You have two context managers, nested:
# session() ← outer context manager
# get_connection() ← inner context manager  
# with - exists so resources are opened, used, and always cleaned up, even if something crashes.
     





# yield 
# can also mean coming back to clean up.
# So “coming back” is really about letting the with block finish and trigger cleanup, 
# not about actually running more lines of code in your function.
# Analogy for clarity
# Think of it like lending a library book:
# You give the book (yield) → the reader reads it
# Reader finishes → you “resume” → you put the book back on the shelf  
# 
# # 
# I’ll trace where monster_resort.db is configured and used so I can explain 
# where it lives, how it’s created, and how the app reads/writes to it.

# Next I’ll quickly inspect how DatabaseManager is instantiated so I can tie 
# monster_resort.db into the app startup and request flow.

# Read main.py, lines 60 to 140

# Here’s how monster_resort.db “exists” and how the app uses it:

# What it is / where it lives

# It’s a SQLite database file on disk, defaulting to a path derived from:
# Settings.database_url = "sqlite:///./monster_resort.db" in config.py:20-32.
# DatabaseManager then uses settings.sqlite_db if present, otherwise falls back 
# to "monster_resort.db" in database.py:69. So:
# If you don’t set a dedicated sqlite_db attribute on Settings, the DB file is 
# monster_resort.db in the current working directory when the app starts.
# If in future you add sqlite_db to Settings, that path will override the default.
# When and how it is created

# On app startup, build_app() in main.py:96-120 does:
# settings = get_settings()
# db = DatabaseManager(settings)
# DatabaseManager.__init__ in database.py:63-76:
# Resolves self.db_path (usually monster_resort.db).
# Ensures the parent directory exists (mkdir(parents=True, exist_ok=True)).
# Calls _init_db(), which:
# Opens a connection to self.db_path.
# Executes all DDL statements in the DDL list to create tables and indexes if they don’t exist.
# Maintains a schema_migrations table and records the current SCHEMA_VERSION (currently 3).
# Logs DEBUG [DB]: Database initialized at {self.db_path} on success.
# What schema it contains (high level)

# Defined by DDL in database.py:15-53:
# schema_migrations: tracks schema version and when it was applied.
# sessions: conversational sessions (session_id, timestamps, summary, metadata).
# messages: individual chat messages per session.
# bookings: guest bookings (public booking_reference, guest details, dates, status, etc.).
# Indexes on messages.session_id, bookings.session_id, and bookings.booking_reference 
# for fast lookups.
# How reads/writes operate

# All DB access goes through context-managed connections in database.py:78-94:
# get_connection():
# Opens sqlite3.connect(self.db_path, check_same_thread=False).
# Sets conn.row_factory = sqlite3.Row so query results act like dicts (access by column name).
# Yields the connection to a with block.
# On normal exit: conn.commit().
# On error: conn.rollback() and raises DatabaseError.
# Always closes the connection in finally.
# session() is just a thin wrapper that reuses get_connection() as a higher-level context.
# Example operations:
# create_booking(...) in database.py:96-149:
# Generates an 8-character booking_reference.
# Inserts a row into bookings with that reference and other details.
# get_booking(booking_id) in database.py:151-168:
# Looks up a booking either by booking_reference or by session_id using a single query.
# How it participates in the running app

# In build_app():
# db = DatabaseManager(settings) wires this DB into other components:
# MemoryStore(db=db) uses it for session/message storage (conversation memory).
# Tool registry (make_registry) uses db for booking-related tools exposed to the LLM.
# When you hit /chat (as in your curl), the request flows through:
# Chat handler → memory tools → db.session() / db.get_connection() → SQLite file monster_resort.db.
# In short: monster_resort.db is the central on-disk SQLite store for sessions, 
# messages, and bookings; it’s created/initialized automatically at app startup 
# via DatabaseManager, and each request that touches state uses short-lived, safely-managed connections into that file.          

In [ ]:
# MAIN.PY 
# remote control. send message. get reply 

def _build_router(settings) -> ModelRouter | None:
def build_app() -> FastAPI:
 async def _agent_reply(session_id: str, user_text: str) -> dict:
 async def chat(payload: dict, _: str = Depends(jwt_or_api_key)):


from fastapi import FastAPI
from app.logging_utils import setup_logging
from app.config import get_settings
from app.advanced_rag import AdvancedRAG
from app.memory import MemoryStore
from app.pdf_generator import PDFGenerator



logger = setup_logging()

def build_app() -> FastAPI:
    settings = get_settings()
    app = FastAPI(title=settings.app_name, version="1.0.0")

    db = DatabaseManager(settings)
    pdf = PDFGenerator(settings.pdf_output_dir)

    rag = AdvancedRAG(
        settings.rag_persist_dir,
        settings.rag_collection,
        embedding_model=getattr(settings, "embedding_model", "all-MiniLM-L6-v2"),
        reranker_model="BAAI/bge-reranker-base",
    )

    memory = MemoryStore(db=db)
    registry = make_registry(
        db=db, pdf=pdf, rag_search_fn=lambda query, k=5: rag.search(query=query, k=k)
    )

def build_app() -> FastAPI:
   settings = get_settings()

def build_app() -> FastAPI:
  settings = get_settings()
  app = FastAPI(title=settings.app_name, version="1.0.0")
  db = DatabaseManager(settings)
  pdf = PDFGenerator(settings.pdf_output_dir)
  rag = AdvancedRAG()
  memory = MemoryStore(db)






# @app.post("/chat")
# async def chat(payload: dict, _: str = Depends(jwt_or_api_key)):

# @profile
# async def _agent_reply(session_id: str, user_text: str) -> dict:

# def _build_router(settings) -> ModelRouter | None:



# Before chat() runs:
# FastAPI calls jwt_or_api_key
# If it fails → request is rejected (401 / 403)
# If it succeeds → chat() is allowed to run
# So Depends(jwt_or_api_key) is basically:
# “You must pass authentication to enter.”



# 4️⃣ Why not just omit the parameter?
# Because FastAPI only runs dependencies if you declare them.
# This won’t work:
# async def chat(payload: dict):
#     jwt_or_api_key()  # ❌ FastAPI won't treat this as a dependency
# You’d lose:
# automatic error handling
# request context
# security integration
# OpenAPI docs
# Depends tells FastAPI:
# “This function is part of the request lifecycle.”



# Absolutely — let’s zoom out one level. A session is basically 
# “a stretch of time where things are treated as connected.”
# Here are 10 intuitive analogies for what session means:
# 1️⃣ A phone call 📞
# Everything said during the call is part of the same session. Hang up = session ends.
# 2️⃣ Sitting at a café table ☕
# While you’re at the table, the staff knows it’s your order. You leave → table resets.
# 3️⃣ A browser tab that stays open 🌐
# As long as the tab is open, the site remembers context. Close it → new session next time.
# 4️⃣ A workout at the gym 🏋️
# One visit, many exercises. Leaving the gym ends the session.
# 5️⃣ A movie night 🎬
# You might pause, rewind, talk — but it’s still the same viewing until you stop.

In [ ]:
# RAG.PY

# Retrieve And Generate 
# The robot first searches its library to find the best pages about your question.
# Then the robot uses those pages to make a nice answer just for you.
# Not just the same words
# But the same meaning
# So even if a page says:
# “Cats make vibrating sounds when happy”
# The robot still knows:
# 👉 “Oh! That’s about purring!”
# The robot reads those pages and explains it in its own words.
# Vector-based RAG is:
# A smart robot that turns questions into numbers, 
# finds the most similar information, and then explains it nicely.

# COLLECTION = STASH

# example = "the cat sat on the wall"
# documents = ["the cat sat on the wall"]
# metadatas = [{"source": "example"}]
# ids = ["a3f1c9d2-..."]
# Behind the scenes, Chroma then:
# Turns "the cat sat on the wall" into a vector (numbers)
# Stores:
# the text
# the vector
# the metadata
# the ID
# You don’t see that part — it’s internal.

import profile
from typing import Iterable
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from app.cache_utils import cache_response

class VectorRAG:
    def __init__(self, collection, persist_path, embedding_model: str = ""):
        self.collection_name = collection 
        self.persist_path = persist_path
        self.embedding_model = embedding_model
        self.chrome_client = chromadb.PersistentClient() 

        self.collection = self.chrome_client.get_or_create_collection(
            name=collection,
            embedding_function=""
        )

    def ingest_text(self, text: Iterator[str]) -> int:
        pass

    def ingest_fodler(self, folder) -> int:
        pass  

    def search(self, query: str, k: int = 5) -> dict: 
        results = self.collection.query(query_texts=[query], n_results=k)
        documents = results.get("documents", [[]])[0]
        pass  
        


    def search(self, query: str, k: int = 5) -> dict:
        try:
            results = self.collection.query(query_texts=[query], n_results=k)
            docs = results["documents", [[]]][0]
            metadatas = results["metadatas", [[]]][0]
            scores =results["distances", [[]]][0]

            out = [
                {"doc": doc, "metadata": meta, "score": score}
                for doc, meta, score in zip(docs, metadatas, scores)
            ]



        except:
            print()    



@profile
@cache_response(ttl=300)
def search(self, query: str, k: int = 5) -> dict:
    try:
        results = self.collection.query(query_texts=[query], n_results=k)
        docs = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        scores = results.get("distances", [[]])[0]

        out = [
            {"text": doc, "meta": meta, "score": float(score)}
            for doc, meta, score in zip(docs, metadatas, scores)
        ]

        if out:
            RAG_HIT_COUNT.labels(query=query).inc()

        return {"ok": True, "query": query, "results": out}
    except Exception as e:
        logger.error(f"ChromaDB search failed: {e}")
        raise AIServiceError(f"ChromaDB search failed: {e}")
 





# 1️⃣ @profile — “Measure how this function behaves”
# This usually means:
# Measure how long the function takes
# Or measure CPU / memory usage
# Or log performance stats
# Plain-English meaning:
# “Keep an eye on this function and record how it performs.”
# Analogy
# It’s like putting a stopwatch and heart monitor on the function every time it runs.
# 2️⃣ @cache_response(ttl=300) — “Save the result for 5 minutes”
# “Keep this cached result for 300 seconds, then throw it away.”
# This decorator:
# Caches the result of search
# If the same inputs are used again, it returns the saved result
# ttl=300 means Time To Live = 300 seconds (5 minutes)


# In a search function, k usually means:
# “Return the top k results”
# So this says:
# By default → return 5 results
# But allow callers to override it

In [ ]:
# ADVANCED RAG 
# # BM25 components
# self.corpus: List[str] = []
# self.bm25: BM25Okapi = None

# # Reranker (lazy loading to save memory)
# self.reranker: CrossEncoder = None
# self.reranker_model = reranker_model


class AdvancedRAG(VectorRAG):
    def __init__(self, collection, persist_path, embedding_model: str = "", reranker_model: str = ""):
        super().__init__(collection, persist_path, embedding_model)

        self.corpus: List[str] = []
        self.bm25: BM250kapi = None 

        self.corpus: List[str] = []
        self.bm25: Optional[BM250kapi] = None 

        # That’s what this line does.
        # tokenized_corpus = your documents broken into individual words
        # BM25Okapi(...) = a search engine algorithm that studies those words
        # self.bm25 = your trained search tool
        
        self.reranker: CrossEncoder = None # super smart judge. CrossEncoder = the type of judge (a specific AI model)
        self.rerenker_model = reranker_model # name of which model to load



    def _dense_search(self, query: str, k: int = 20) -> List[Tuple[str, float]]:
        results = super().search(query, k) # dict 

        return [
            (r["text"], r["score"])
            for r in results.get("results", [])
        ]
        

    def _dense_search(self, query: str, k: int = 20) -> List[Tuple[str, float]]:
      results = super().search(query, k=k)
      return [
          (r["text"], r["score"]) 
        for r in results.get("results", [])
        ]   


    def _bm25_search(self, query: str, k: int = 20) -> List[Tuple[int, float]]:
        if self.bm25 is None: # no trained librarian 
            return []
        tokenized_text = query.lower().split()
        scores = self.bm25.get_scores(tokenized_text)



    def _bm25_search(self, query: str, k: int = 20) -> List[Tuple[int, float]]:
        if self.bm25 is None:
            logger.warning("BM25 index not built, returning empty results")
            return []

        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)



class AdvancedRAG(VectorRAG):
        def __init__(self, collection, persist_path, embedding_model = "", reranker_model = ""):
             super().__init__(collection, persist_path, embedding_model)

             self.corpus: List[str] = []
             self.bm25: Optional[BM250kapi] = None 
             self.reranker: Optional[CrossEncoder] = None 
             self.reranker_model = reranker_model



        # Line 1 → Break your question into Lego pieces.
        # Line 2 → Check every book and give it a relevance score based on those pieces.
        # No books are filtered out yet — they’re just graded.
        # Later, the system says:
        # “Show me the top 20 highest-graded books.”

        # Get top-k with positive scores
        import numpy as np

        top_indices = np.argsort(scores)[::-1][:k]
        # 🚀 Practical Example
        # [::-1]
        # This reverses the array.
        # scores = np.array([88, 92, 79, 93, 85])
        # k = 3

        # top_indices = np.argsort(scores)[::-1][:k]
        # print(top_indices)
        # Output:
        # array([3, 1, 0])
        # Meaning:
        # 93 (index 3)
        # 92 (index 1)
        # 88 (index 0)

        # Imagine you have 10,000 resumes stored in a system.
        # Instead of:
        # Filtering resumes where “Python” appears
        # You:
        # Score every resume based on how relevant it is to “Senior Python Backend Engineer”
        # Then sort by highest score
        # Then take the top 20
        # That scoring step is the search.
        return [
            (int(idx), float(scores[idx])) for idx in top_indices if scores[idx] > 0
        ]

    def _dense_search(self, query: str, k: int = 20) -> List[Tuple[str, float]]:
        results = super().search(query, k=k)
        return [(r["text"], r["score"]) for r in results.get("results", [])]   


# self.corpus: List[str] = []
# self.bm25: BM25Okapi = None
# self.reranker: CrossEncoder = None
# self.reranker_model = reranker_model


# Think of it like searching for the best answer in a huge library.

### 🏛️ Step 1: Two different librarians help you search (Hybrid Search)

# You ask: *“How do I improve retrieval accuracy?”*

# * **Librarian #1 (BM25 – keyword search)**
#   This librarian looks for books that contain the exact words you said.
#   If your question includes “retrieval” and “accuracy,” they find books with those exact words.

# * **Librarian #2 (Dense embeddings – semantic search)**
#   This librarian doesn’t just look for the same words — they look for the same *meaning*.
#   Even if a book says “improving search precision” instead of “retrieval accuracy,” they 
# still find it because they understand the concept.

# 👉 Hybrid search means **both librarians search at the same time**, so you don’t miss 
# useful results just because different wording was used.



### 🔀 Step 2: They combine their findings (Reciprocal Rank Fusion)

# Now you have two piles of books — one from each librarian.

# Instead of choosing one pile over the other, you:

# * Look at how highly each librarian ranked each book
# * Combine their rankings intelligently

# If both librarians ranked a book highly, it moves near the top.
# If only one ranked it highly, it still has a chance — but not as strong.

# That smart merging method is called **Reciprocal Rank Fusion (RRF)**.



### 🎯 Step 3: A senior expert double-checks the best results (Cross-Encoder Reranking)

# Now you have a shortlist of promising books.

# Instead of just trusting titles or summaries, a senior expert:

# * Carefully reads your question
# * Carefully reads each shortlisted book
# * Scores how well each book truly answers your question

# This deeper comparison is the **cross-encoder reranker** (BGE model).
# It’s slower but much more precise — so it’s only used on the top results.

# This is called **two-stage reranking**:

# 1. Fast search to narrow things down
# 2. Careful evaluation to pick the best ones

# ---

# ### 🏭 Production-ready

# Finally:

# * **Caching** = If someone asks the same question again, you don’t redo all 
# the work — you reuse results.
# * **Error handling** = If one librarian fails, the system doesn’t crash; it still 
# works gracefully.

# ---

# ### In short:

# It’s like:

# * 🔎 Two smart search methods working together
# * 🤝 A smart way of combining their rankings
# * 👨‍⚖️ An expert judge re-evaluating the finalists
# * 🛠️ Built to work reliably in real-world conditions

# All to make sure you get the most accurate answers possible.



In [ ]:
# ADVANCED RAG 
# # BM25 components
# self.corpus: List[str] = []
# self.bm25: BM25Okapi = None

# # Reranker (lazy loading to save memory)
# self.reranker: CrossEncoder = None
# self.reranker_model = reranker_model


class AdvancedRAG(VectorRAG):
    def __init__(self, collection, persist_path, embedding_model = "", rerenker_model=""):
        super().__init__(collection, persist_path, embedding_model)
        self.reranker: CrossEncoder = None 
        self.reranker_model = rerenker_model

    def _load_reranker(self):
        if self.reranker is None:
            self.reranker = CrossEncoder(self.reranker_model )    



# A reranker takes a list of results and reorders them to put the best ones first. 
# Here are 10 very simple analogies:
# Sorting exam papers so the highest scores are on top
# Reordering search results so the most relevant link is first
# A judge rearranging contestants from best to worst

In [ ]:
# LANGCHAIN.PY 
# VectorRAG and LangChainRAG are just different librarians 

# Mental model (simple)
# List = a box you can open many times 📦
# Iterator = a conveyor belt that passes once 🛤️
# This function needs a box, not a belt.

from langchain_huggingface import HuggingFaceEmbeddings


class LangChain:
    def __init__(
        self, 
        collection, persist_dir, 
        embedding_model: str = "", 
        chunk_size: int = 500,
        chunk_overlap: int = 50
    ):
        from langchain_community.vectorstores import Chroma

        self.colleaction_name = collection 
        self.persist_dir = persist_dir
        self.embedding_model = embedding_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.embeddings = HuggingFaceEmbeddings(model_name= embedding_model)

        self.vector_store = Chroma(
            collection_name=collection,
            embedding_function=self.embeddings,
            persist_directory=self.persist_dir
        )


    def ingest_texts(self, texts: List[str], *, source: str = "manual") -> int:
        splitter = REC()

        all_chunks = []
        all_metadata = []
        all_id = []

        for text in texts:
            chunks = splitter.split_text(text)
            ... 

    def ingest_texts(self, texts: List[str], *, source: str = "manual") -> int:
        splitter = RC()

        all_chunks = []
        all_metadat = []
        all_id = [] 

        for text in texts:
            chunks = splitter.split_text(text)
            for chunk in chunks:
                all_chunks.append(chunk)




    

    def ingest_texts() -> int:
        ...
    def ingest_folder() -> int:
        ...
    def search() -> dict:
        ...           
        


# chunk_overlap = 50
# This means:
# “When I cut the strips, let 50 characters repeat between neighbors.”
# So instead of clean cuts, you get:
# Chunk 1: characters 1–500
# Chunk 2: characters 451–950
# Chunk 3: characters 901–1400
# Those overlapping 50 characters act like glue 🧠
# Why overlap exists (important!)
# Without overlap, you risk this:
# "...the cat sat on the"
# | cut here |
# "wall and looked around..."
# Now:
# One chunk has “the cat sat on the”
# The next has “wall and looked around”
# Neither chunk alone contains the full idea.
# Overlap makes sure:
# Sentences
# Thoughts
# Definitions
# Don’t get chopped in half.
# Map this to your “cat on the wall” example 🐈
# Text:
# "the cat sat on the wall and looked at the birds outside"
# With:
# chunk_size = 20
# chunk_overlap = 5
# You might get:
# Chunk 1: "the cat sat on the"
# Chunk 2: "on the wall and"
# Chunk 3: "and looked at the"
# Chunk 4: "at the birds outside"
# Each chunk shares a bit with the next.        

In [ ]:
# SECURITY.PY 
# so an api key is basically "pls allow me to work with you"


class APIKeyManager:
    def __init__(self):
        print()
    def create_key(self, user_id: str, expires: int=90) -> str:
        print()

    def create_key(self, user_id: str, expires: int = 90) -> str:
        ... 
    
    def verify_key(self, api_key: str) -> Optional[str]:
        ...



# This file handles secure access to an API. 
# It creates and checks API keys, tracks how they’re used, 
# blocks invalid or expired keys, enforces proper login headers, 
# and limits how often requests can be made to prevent abuse.

# Bearer token and API Key


# Exactly. As the head of this tech company, you’ve just described a perfect Audit & Monitoring 
# System. You aren't just letting people in; you're gathering "Business Intelligence" on how 
# your office operates.
# Here is how your Master Dashboard would actually look and function using that data:
# 🖥️ The Master Dashboard View
# Because you are using API Keys for the cubicles and Bearer Tokens for the employees, 
# your dashboard can answer two different, vital questions:
# 1. The "Infrastructure" View (API Key)
# * What you see: "Cubicle #4 (North Wing) has made 15,000 requests today."
# * The Insight: You notice Cubicle #4 has a much higher volume than the others. Is the computer there malfunctioning? Is a script running in a loop?
# * The Action: You can Rate Limit that specific API Key. This means Cubicle #4 is capped at 500 requests per hour so it doesn't crash your whole server, regardless of who is sitting there.
# 2. The "HR/Productivity" View (Bearer Token)
# * What you see: "Employee 'Sarah Jenkins' logged in at 9:00 AM and "
# "accessed 'Client Financial Records'."
# * The Insight: You can see exactly what data is being touched.
# * The Action: If Sarah leaves the company, you don't need to change the cubicle's API Key. 
# You simply Revoke her Bearer Token. She can sit at the same desk, but 
# the "Embassy Clerk" (the server) will no longer accept her expired visitor slip.


# print(verify_key("my-secret-key"))  # -> 'user123'
# print(verify_key("wrong-key"))      # -> None


In [ ]:
# VALIDATION.PY 

import bleach
from email_validator import validate_email as _validate_email, EmailNotValidError

def santitize(text) -> str:
    bleach.clean(text=text, tags=[], strip=True)


def validate_email(text: str) -> str:
    try:
        v = _validate_email(text)
        email = v.email.lower()
    except EmailNotValidError as e:
        raise ValidationError(f"invalid email format {e}")
    


def sanitize(text: str) -> str:
    bleach.clean(text, tags=[], strip=True)

def validate_email(text: str) -> str:
    try:
        v = _validate_email(text)
        email = v.email.lower*()
        return email 
    except EmailNotValidError as e:
        raise ValidationError(f"invalid email format: {e}")    



# This file checks and cleans user input to keep an app safe and 
# reliable. It validates messages, names, dates, bookings, and emails, 
# removes harmful content, blocks suspicious attacks, and ensures everything 
# follows sensible rules before being accepted.

def sanitize_html(text: str) -> str:  
def validate_message(message) -> str:
def validate_guest_name(name: str) -> str:
def validate_date(date_str: str, field_name: str = "Date") -> str:
def validate_email(email: str) -> str: (2)

def validate_booking(
    guest_name: str, room_type: str, check_in: str, check_out: str, guests: int
) -> dict:
    
# 


In [ ]:
# CACHE.PY
# cache_utils 

# CACHE_MAX_SIZE = 256 means the cache can store up to 256 items. What counts as 
# an “item” here is one key-value pair in the cache:
# Key: a unique label representing a specific function call and its arguments (from _make_key).
# Value: the result of that function call.


class TTLCache:
    def __init__(self, max_size: int = 256, ttl: int = 300):
        print()


class TTLCache:
    ...

def cache_response():
    return decorator 


class TTL:
    print() 

def cache_response():
    def decorator(func: Callable):
        ...
        return decorator     

# we now use the decorator 

# This code adds a smart “memory” to functions. 
# It saves recent results so repeated calls return instantly instead of recalculating. 
# The memory is limited in size, automatically forgets old results after a few minutes, 
# and is safe to use with multiple threads.


In [ ]:
# MONITORING.PY 




def track_metrics(endpoint_name):
    print()

In [ ]:
# app/profile_utils.py
# stop watch 



def profile(func):
    if print(""):
        print()
    else:
        print()    


def profile(func):
    if print(""):
        print()
    else:
        print("")            


def profile(func):
    if print():
        print()
    else:
        print("")    

In [ ]:
# LOGGING
# logging_utils.py

class Stero(Exception):
    pass 


class JSONFormatter():
    def format(self, record):
        ...

class Stero(Exception):
    pass      

def setup_logging(log_level="INFO"):
    pass 


class JSONFormatter(logging.Formatter):
    def format(self, record):
        ...

logger = setup_logging()


In [ ]:
# profile_utils.py
# PROFILE UTILS 


import asyncio
import functools
import logging
import time

def profile(func):
    if asyncio.iscoroutinefunction(func):
        async def async_wrapper():
            ...
    else:
        def sync_wrapper():
            ...

def profile(func):
    if asyncio.iscoroutine(func):
        async def asyncwrapper():
            ...
    else:
        def syncwrapper():
            ...      

def profile(func):
    """Profile execution time for both sync and async functions."""
    logger = logging.getLogger("monster_resort")

    if asyncio.iscoroutinefunction(func):
        @functools.wraps(func)
        async def async_wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = await func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            logger.info(f"[PROFILE] {func.__name__} took {elapsed:.4f}s")
            return result

        return async_wrapper
    else:
        @functools.wraps(func)
        def sync_wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            logger.info(f"[PROFILE] {func.__name__} took {elapsed:.4f}s")
            return result

        return sync_wrapper





# It’s like having a stopwatch-wearing assistant who waits outside every room you enter. 
# No matter if you're taking a quick nap or doing a long workout, they click the timer '
# 'when you start and shout your total time the moment you're finished.

# How this works in your code

# The @profile decorator acts as that assistant by "wrapping" your original 
# function. It doesn't care if the task is synchronous (happening right now) or 
# asynchronous (waiting on something else); it just ensures the clock starts before the work 
# begins and stops only when the work is officially done.

In [ ]:
# HALLUCINATION.PY 
# Semantic similarity is basically a way to measure how close in meaning 
# two pieces of text are, even if they use different words.



if score >= high_threshold:
    print("Highly likely hallucination 🚨")
elif score >= medium_threshold:
    print("Possibly a hallucination ⚠️")
else:
    print("Likely accurate ✅")

class ConfidenceResult:
    print()


class HallucinationDetector:
    def __init__(self):
        print()

    def score_response() -> ConfidenceResult:
        return ConfidenceResult()
    

HallucinationDetector
HallucinationDetector
HallucinationDetector
HallucinationDetector
HallucinationDetector



class HallucinationDetector:
    def __init__(self, high_threshhold: float = 0.7, medium_threshhold: float = 0.4):
        self.high_threshhold = high_threshhold
        self.medium_threshhold = medium_threshhold
        self.model = None 


    def score_response() -> ConfidenceResult:
        ...    



class HallucinationDetector:
    def __init__(self, high: float=0.7, medium: float=0.4):
        self.high = high 
        self.medium = medium

    def score_response() -> ConfidenceResult:
        pass     

# overlap = self._compute_context_overlap(response_text, rag_contexts)
# semantic = self._compute_semantic_similarity(response_text, rag_contexts)
# attribution = self._compute_source_attribution(response_text, rag_contexts) 
# Think of these three as **different ways to check how well your model’s answer 
# is using the RAG context**.

## 1️⃣ Context overlap

# ```python
# overlap = self._compute_context_overlap(response_text, rag_contexts)
# ```

# **What it asks:**

# > “Did the answer reuse the *actual words* from the retrieved documents?”

# **Simple explanation:**

# * It looks for **shared words or phrases** between:

#   * the model’s response
#   * the RAG context chunks
# * More shared words → higher overlap score

# **Example:**
# RAG context:

# > “Paris is the capital of France.”

# Model response:

# > “Paris is the capital of France.”

# ➡️ **Very high overlap**

# Model response:

# > “France’s capital city is Paris.”

# ➡️ **Lower overlap** (same meaning, different words)

# **What it’s good for:**

# * Detecting **copying / grounding**
# * Catching hallucinations where the model invents info not present in context

# **What it’s bad at:**

# * Penalizes paraphrasing
# * Doesn’t understand meaning, only text similarity

# ---

# ## 2️⃣ Semantic similarity

# ```python
# semantic = self._compute_semantic_similarity(response_text, rag_contexts)
# ```

# **What it asks:**

# > “Does the answer *mean the same thing* as the context, even if the words differ?”

# **Simple explanation:**

# * Uses embeddings (vector meaning)
# * Checks if the **idea** in the response matches the **idea** in the context

# **Example:**
# Context:

# > “The Eiffel Tower was completed in 1889.”

# Response:

# > “The Eiffel Tower finished construction in the late 19th century.”

# ➡️ **High semantic similarity**, even though wording is different

# **What it’s good for:**

# * Measuring **understanding**
# * Allowing paraphrases
# * Evaluating answer quality more fairly

# **What it’s bad at:**

# * Can still score high if the answer is **vaguely related but incorrect**
# * Doesn’t prove the info came *from* the context

# ---

# ## 3️⃣ Source attribution

# ```python
# attribution = self._compute_source_attribution(response_text, rag_contexts)
# ```

# **What it asks:**

# > “Can we trace each claim in the answer back to the retrieved sources?”

# **Simple explanation:**

# * Checks whether statements in the response are:

#   * supported by
#   * explicitly or implicitly grounded in
#     the RAG context
# * This is about **trust and provenance**

# **Example:**
# Context:

# > “Aspirin reduces inflammation and pain.”

# Response:

# > “Aspirin reduces inflammation and pain, and also cures cancer.”

# ➡️ The first claim is attributed
# ➡️ The second claim is **unattributed** (hallucination)

# **What it’s good for:**

# * Detecting hallucinations
# * Evaluating RAG faithfulness
# * Safety & reliability scoring

# **What it’s hard to do:**

# * This is the **most complex** metric
# * Often involves sentence-level matching or NLI models

# ---

# ## 🧠 How they work together (intuition)

# | Metric              | Checks       | Catches                 |
# | ------------------- | ------------ | ----------------------- |
# | Context overlap     | Same words   | Fabrication, copying    |
# | Semantic similarity | Same meaning | Paraphrased correctness |
# | Source attribution  | Same source  | Hallucinations          |

# **Best practice:**

# > High semantic similarity + reasonable overlap + strong attribution
# > = **good RAG answer**



# // why do some method names begin with a _ 
# “This is intended for internal use only.”
# It’s not meant to be called from outside the class or module.
# Python does not enforce this — it’s purely a convention to communicate intent.    

In [ ]:
# CACHE EXPLAINED 


# Sure! Let's explain your code using a **simple, real-life analogy**. Think of it like a **kitchen pantry with snacks**. 🍪

# ---

# ### **1. The Pantry (Cache)**

# Your `TTLCache` is like a **pantry** where you store snacks (data) that you want to grab quickly instead of going to the store (recomputing data).

# * `CACHE_MAX_SIZE = 256` → Your pantry can only hold **256 snack items**. If it gets full, you throw out the **oldest snack** to make space.
# * `CACHE_TTL_SECONDS = 300` → Each snack has an **expiry time** of 5 minutes. After that, it’s stale and must be thrown away.

# ---

# ### **2. Putting snacks in the pantry (`set`)**

# When you add a snack:

# * If it’s already in the pantry, you move it to the front (so it’s the **most recently used**).
# * You mark it with an **expiry time** (when it will go bad).
# * If the pantry is full, you remove the **oldest snack** (least recently used).

# ---

# ### **3. Taking snacks out (`get`)**

# When you want a snack:

# * If it’s not in the pantry → go get it fresh from the store (compute it from the function).
# * If it’s in the pantry but expired → throw it away and get a fresh one.
# * If it’s in the pantry and still good → take it, and move it to the front (because it’s now **recently used**).

# ---

# ### **4. Making a label for each snack (`_make_key`)**

# Every snack (function call) gets a **unique label** so you can find it in the pantry:

# * The label is made from the **function name + all the ingredients** (arguments).
# * Think of it as writing the recipe on the snack so you don’t confuse it with others.

# ---

# ### **5. The magical pantry helper (`cache_response`)**

# The decorator `cache_response` is like having a **helper who automatically puts snacks in the pantry** for you:

# * When you call a function, the helper first **checks the pantry**.
# * If the snack is already there and still fresh → returns it immediately.
# * If it’s not there or expired → makes a fresh snack, puts it in the pantry, and gives it to you.

# ---

# ### **TL;DR Analogy**

# | Code Concept        | Snack Analogy                                |
# | ------------------- | -------------------------------------------- |
# | TTLCache            | Pantry for snacks                            |
# | `set()`             | Put a snack in the pantry, mark expiry       |
# | `get()`             | Take a snack out if fresh, otherwise get new |
# | `CACHE_MAX_SIZE`    | Pantry size limit                            |
# | `CACHE_TTL_SECONDS` | Snack expiry time                            |
# | `_make_key()`       | Label snacks so you can find them            |
# | `cache_response`    | Helper who automatically uses the pantry     |

# ---

# If you want, I can also **draw a tiny visual diagram** showing this pantry analogy with snacks moving in and out—makes it super easy to “see” how your cache works.

# Do you want me to do that?


In [ ]:
# # ADVANCED RAG DOC FOR READING 

# Perfect — let’s walk **one concrete story** all the way through, using **“the cat sat on the wall”**, and I’ll show **exactly where reranking lives in RAG**.

# No abstractions. Just steps.

# ---

# ## The knowledge we stored earlier 📚

# In your vector database, you have these texts:

# 1. **“the cat sat on the wall”**
# 2. “the dog slept on the floor”
# 3. “birds sit on trees”

# Each one:

# * was turned into a vector
# * stored with an ID
# * maybe has metadata like `source="example"`

# ---

# ## You ask a question 🗣️

# ```text
# "where did the cat sit?"
# ```

# This kicks off **RAG**.

# ---

# ## Step 1: Retrieval (vector search) 🔍

# The system:

# 1. Turns your question into a vector
# 2. Asks Chroma:

#    > “Give me the top 3 closest texts”

# Chroma might return (based on vector similarity):

# | Rank | Text                         |
# | ---- | ---------------------------- |
# | 1    | “birds sit on trees”         |
# | 2    | “the cat sat on the wall”    |
# | 3    | “the dog slept on the floor” |

# ⚠️ Notice:

# * Vector search is **fast**
# * But not always perfectly precise
# * It sometimes mixes up “sit” vs “sat”, “where” vs “on”

# At this point, results are **rough**.

# ---

# ## Step 2: Reranking (THIS IS THE RERANKER) 🏆

# Now the **reranker model** steps in.

# The reranker:

# * Takes the **question**
# * Takes each retrieved text
# * Scores them **one-by-one** for relevance

# It asks, in effect:

# > “Does this sentence actually answer the question?”

# ### Reranker scores

# | Text                         | Relevance score |
# | ---------------------------- | --------------- |
# | “the cat sat on the wall”    | ⭐⭐⭐⭐⭐           |
# | “birds sit on trees”         | ⭐⭐              |
# | “the dog slept on the floor” | ⭐               |

# Now the order becomes:

# 1. **“the cat sat on the wall”**
# 2. “birds sit on trees”
# 3. “the dog slept on the floor”

# This is **reranking**.

# ---

# ## Step 3: Generation (answering) ✍️

# Only **after reranking**, the LLM sees the context:

# ```text
# Context:
# - the cat sat on the wall
# - birds sit on trees
# ```

# And answers:

# > “The cat sat on the wall.”

# ---

# ## Where reranking fits in RAG (big picture)

# ```text
# Your question
#    ↓
# Vector search (fast, fuzzy)
#    ↓
# RERANKER (slow, precise)  ← ⭐ HERE
#    ↓
# Top results
#    ↓
# LLM generates answer
# ```

# ---

# ## Why reranking exists at all 🤔

# Vector search:

# * Is great at “similar meaning”
# * Is bad at “exactly answers the question”

# Rerankers:

# * Are slower
# * But much smarter
# * And much more precise

# So the combo is:

# * **Retrieve broadly**
# * **Rerank carefully**

# ---

# ## Cat analogy 🐈

# * Vector search = “grab anything cat-related”
# * Reranker = “okay, but which one actually answers *this* question?”
# * LLM = “say it nicely”

# ---

# ## One-sentence takeaway 🧠

# > Reranking is the **second brain pass** that fixes the rough ordering from vector search before the LLM answers.

# If you want next:

# * I can show this as pseudo-code
# * Or explain why rerankers are cross-encoders
# * Or help you decide when reranking is worth the cost


In [ ]:

#   self.persist_dir = persist_dir
#         self.collection_name = collection
#         self.embedding_model = embedding_model

#         # Initialize ChromaDB client
#         self.chroma_client = chromadb.PersistentClient(path=persist_dir)

#         # Create or get collection with HuggingFace embeddings
#         self.collection = self.chroma_client.get_or_create_collection(
#             name=collection,
#             embedding_function=SentenceTransformerEmbeddingFunction(
#                 model_name=self.embedding_model
#             ),
#         )




# chromedb = bookshelves 
# embeddings = book topics converted to numbers 
# rag = finding the right book to answer your question 
    



# Library and Dewey Decimal
# ChromaDB = library shelves
# Embeddings = book topics converted into numbers
# RAG = finding the right book to answer your question
# Analogy: You ask the librarian (RAG) a question. They look up the 
# book number (embedding) in the shelves (ChromaDB) and give you the right book


In [ ]:
# LLMPROVIDERS.PY 
# llm_providers.py

# Conceptually
# Think of it like:
# @abstractmethod → puts a red “MUST OVERRIDE” sticker on the function.
# @property → turns that function into a “computed attribute”.
# So subclasses are required to implement a property called name.

from abc import ABC, abstractmethod
from typing import List, Optional


class LLMProvider(ABC):
    @abstractmethod
    def chat() -> LLMProvider:
        ...

    @abstractmethod
    def translate_tool_schemas(self, openai_schemas: List[dict]) -> List[dict]:
        ... 

    @property
    @abstractmethod
    def name(self) -> str:
        ...         


class LLMProvider(ABC):
    @abstractmethod
    async def chat() -> LLMResponse:
        ...

    @abstractmethod
    def translate_openai_schema(self, openai_schema: List[dict]) -> List[dict]:
        ...
    
    @property
    @abstractmethod
    def name(self) -> str:
        ...



class ModelRouter:
    def __init__(self, providers: List[LLMProvider], fallback_enabled: bool = True):
        self.providers = providers 
        self.fallback_enabled = fallback_enabled

    async def chat(
        self, 
        messages: List[LLMMesage], 
        tools: Optional[List[dict]] = None,
        model: Optional[str] = None
    ) -> LLMResponse:
        for provider in self.providers:
            try:
                logger.info(f"starting {provider.name}")
                response = await provider.chat(messages, tools, model)
                logger.info(f"{provider.name} succeeded")
                return response 
            except Exception as e:
                logger.warning(f"llm provider {provider.name} failed: {e}")   
                if not self.fallback_enabled:
                    raise
        raise RuntimeError(f"All LLM providers failed. Last error: {last_error}")        
            
    async def chat(
        self, 
        messages: List[LLMMesage], 
        tools: Optional[List[dict]] = None,
        model: Optional[str] = None
    ) -> LLMResponse:
        for provider in self.providers:
            try:
                logger.info("")
                response = await provider.chat(messages, tools, model)
                return response 
            except Exception as e:
                logger.warning()
                
        raise RuntimeError()           
        



   


class ModelRouter:
    def __init__(self, providers: List[LLMProvider], fallback_enabled: bool = None):
        self.providers = providers
        self.fallback_enabled = fallback_enabled

    async def chat(
            self,
            message: List[LLMMessage],
            tool: Optional[List[dict]] = None,
            model: Optional[str] = None 
        ) -> LLMResponse:
        for provider in self.providers:
            try:
                response = await provider.chat(message, tool, model)
                return response
            except Exception as e:
                print()
        raise RuntimeError()            
        

            
             
        
   

# def __init__(self, providers: List[LLMProvider], fallback_enabled: bool = True):

# async def chat(
#         self,
#         messages: List[LLMMessage],
#         tools: Optional[List[dict]] = None,
#         model: Optional[str] = None,
#     ) -> LLMResponse:



# async def

### **Ordering food in a restaurant**

# * You place your order (`await cook_food()`), then **don’t just 
# stand there** — you chat, scroll your phone, or talk to friends while waiting.
# * `async def` lets the program “do other stuff” while waiting.



### 2️⃣ **Laundry**

# * You put clothes in the washer (`await wash()`), then **do other chores** 
# instead of staring at the machine.
# * The washing machine works in parallel; you’re not blocked.

# ---

### 3️⃣ **Sending an email**

# * You hit “send” (`await email_server.send()`), and instead of freezing, 
# you go back to writing another email.


### 4️⃣ **Downloading files**

# * You start downloading one file (`await download(file1)`), while 
# simultaneously **initiating other downloads**.




SyntaxError: expected ':' (1629893370.py, line 10)

In [ ]:
# memory.py

from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Optional

from .database import DatabaseManager

@dataclass
class MemoryStore:
    dbm: DatabaseManager
    max_mess: int = 12

    def ensure_session(self, session_id: str, metadata: Optional[dict] = None) -> None:
        ...

    def add_message(self, session_id: str, role: str, content: str) -> None:
        ...

    def get_messages(self, session_id: str, limit: int = 50) -> list[dict]:
        ... 

    def get_summary(self, session_id: str) -> Optional[str]:
        ... 

    def _maybe_summarise(self, session_id: str) -> None:
        ... 

    def _cheap_summary(self, lines: list[str]) -> str:
        ... 


@dataclass 
class MemoryStore:
    db: DatabaseManager
    maxmessages: int = 12 

    def ensure_session(self, session_id: str, metadata: Optional[dict] = None) -> None:
        now = datetime.now(timezone.utc).isoformat()

        with self.db.session as conn:
            row = conn.execute().fetchone()

            if row is None:
                print()
            else:
                print()    

    def ensure_session(self, ) -> None:
        now = datetime.now(timezone.utc).isoformat()

        with self.db.session as conn: 
            row = conn.execute().fetchone()

            if row is None:
                print()
            else:
                print()    


    


In [ ]:
# tools.py 

from dataclasses import dataclass
from typing import Any, Callable, Counter, Dict


def describe_tool_for_ai(self) -> dict:
    print(f"[DEBUG] describing tool: {self.name}")
    return {}
# What this function is NOT doing 🚫
# Not calling OpenAI
# Not validating anything
# Not executing the tool
# Not transforming data
# It’s just describing.


# Analogy for Callable[..., Any]:
# A button you can press:
# Maybe it needs arguments
# Maybe it doesn’t
# Maybe it gives something back
# Maybe it doesn’t
# You don’t know. You just press it.



logger.info()
TOOL_CALL_COUNT = Counter()
Toolfn = Callable[..., Any]

# so jar, number, categories 
# Exactly 👍

# In super simple terms:

# * **Jar** → the **Counter** itself (what is keeping track)
# * **Number of beads inside** → the **total count** (how many times something happened)
# * **Categories (stickers)** → the **labels** (how you group the counts)

# So after 1 hour:

# * Jar name: “Candy Count”
# * Total beads in jar: **20**
# * Categories:

#   * Chocolate → 12
#   * Gummy → 8

# And the important rule:
# You can only **add beads**, never remove them.

# That’s exactly how a Prometheus `Counter` works.


logger.info()
TOOL_CALL_COUNT = Counter("jar", "number", ["categories"])
ToolFn = Callable[..., Any]


@dataclass
class Tool:
    name: str 
    desc: str 
    fn: ToolFn

    def to_openai_schema(self) -> dict:
        if self.name == "somefunction":
            return {
                "name": "ans",
                "description": "",
                "parameters": {
                    "type": "object",
                    "properties": {"abc": {"type": "string"}},
                    "required": ""
                }
            }


class ToolRegistery:
    def __init__(self):
        self.tool: Dict[str, Tool] = {}





@dataclass(frozen=True)
class Tool:
    name: str 
    desc: str 
    fn: ToolFn

    def to_openai_schema(self) -> dict:
        if self.name == "":
            return {
                "name": "book_room",
                "description": "parameters",
                "parameters": {
                    "type": "object",
                    "properties": {"a": {"type": "string"}},
                    "required": "a"

                }
            }
        elif self.name == "generate_invoice":
            return {
                "name": "generate",
                "description": "some description",
                "parameters": {
                    "type": "object",
                    "properties": {"a": {"type": "string"}},
                    "required": "a"
                }
            }
        return {}


class ToolRegistry:
    def __init__(self):
        self.tool: Dict[str, Tool] = {}



@dataclass(frozen=True)
class Tool:
    name: str 
    desc: str
    fn: Toolfn 

    def to_openai_schema(self) -> dict:
        if self.name == "":
            return {}
        elif self.name == "":
            return {}
        return {}
    

class ToolRegistry:
    def __init__(self):
        self.tool: Dict[str, Tool] = {}

    def register(self, name: str, desc: str):
        print(f"registering tool with {name} {desc}")

        def decorator(fn: Toolfn):
            self.tool[name] = Tool(name, desc, fn)
            return fn 
        
        return decorator    
    


class ToolRegistry:
    def __init__(self):
        self.tool: Dict = {}
        
    def register(self, name: str, desc: str):
        print()

        def decorator(fn: Toolfn):
            self.tool[name] = Tool(name, desc, fn)
            return fn 
        return decorator 


# @dataclass(frozen=True)
# class Tool:
#     name: str
#     description: str
#     fn: ToolFn
# Callable[[Arg1Type, Arg2Type, ...], ReturnType]
# "..." means ->Callable[[int, int], str]  # a function taking two ints and returning a string


# Love this question — you’re asking for **training wheels**, not production code. Perfect 👍
# Let’s strip *everything* down to the absolute minimum.

# ---

# ## 1️⃣ Super-simple dummy `Counter`

# Forget Prometheus, metrics, labels, all of that.

# ### Practice version

# ```python
# class Counter:
#     def __init__(self):
#         self.count = 0

#     def inc(self):
#         self.count += 1
# ```

# Usage:

# ```python
# TOOL_CALL_COUNT = Counter()

# TOOL_CALL_COUNT.inc()
# TOOL_CALL_COUNT.inc()

# print(TOOL_CALL_COUNT.count)  # 2
# ```

# That’s it.

# * It **counts**
# * Nothing fancy
# * No strings, no labels

# ---

# ## 2️⃣ Super-simple dummy `ToolFn`

# In real code:

# ```python
# ToolFn = Callable[..., Any]
# ```

# Means:

# > “A function that takes anything and returns anything.”

# ### Practice version (no typing)

# ```python
# def tool_fn():
#     print("Tool was called!")
# ```

# Call it:

# ```python
# tool_fn()
# ```

# ---

# ### Practice version (with typing, still simple)

# ```python
# from typing import Callable

# ToolFn = Callable[[], None]
# ```

# Example tool:

# ```python
# def say_hi() -> None:
#     print("Hi!")
# ```

# Now `say_hi` is a `ToolFn`.

# ---

# ## 3️⃣ Put them together (tiny toy example)

# ```python
# class Counter:
#     def __init__(self):
#         self.count = 0

#     def inc(self):
#         self.count += 1


# ToolFn = Callable[[], None]

# TOOL_CALL_COUNT = Counter()


# def my_tool():
#     TOOL_CALL_COUNT.inc()
#     print("Doing tool work")
# ```

# Run:

# ```python
# my_tool()
# my_tool()

# print(TOOL_CALL_COUNT.count)  # 2
# ```

# ---

# ## Mental model 🧠

# * `Counter` = a box with a number
# * `.inc()` = add 1
# * `Callable` = “this thing is a function”
# * `ToolFn` = “a function I can call as a tool”

# ---

# ## One-sentence takeaway

# > These are just **names for very simple ideas** — counting and “this is a function”.

In [ ]:
# Perfect — `logger.info` is another one that feels “magical” until you see the bones 🦴
# Let’s do the **simplest possible practice version**.

# ---

# ## 1️⃣ What `logger.info` really is (plain English)

# `logger.info("something")` just means:

# > “Print a message saying something normal happened.”

# That’s it.

# No magic required.

# ---

# ## 2️⃣ Absolute simplest dummy logger

# ```python
# class Logger:
#     def info(self, message):
#         print(message)
# ```

# Usage:

# ```python
# logger = Logger()

# logger.info("Hello!")
# ```

# Output:

# ```
# Hello!
# ```

# That’s already 80% of the idea.

# ---

# ## 3️⃣ Slightly better (shows levels)

# ```python
# class Logger:
#     def info(self, message):
#         print(f"[INFO] {message}")

#     def warning(self, message):
#         print(f"[WARN] {message}")

#     def error(self, message):
#         print(f"[ERROR] {message}")
# ```

# Usage:

# ```python
# logger = Logger()

# logger.info("Starting app")
# logger.warning("Something looks off")
# logger.error("Something broke")
# ```

# Output:

# ```
# [INFO] Starting app
# [WARN] Something looks off
# [ERROR] Something broke
# ```

# ---

# ## 4️⃣ Map this to your real code 🧠

# Your real line:

# ```python
# logger.info(f"Trying LLM provider: {provider.name}")
# ```

# Is conceptually the same as:

# ```python
# print(f"[INFO] Trying LLM provider: {provider.name}")
# ```

# The real `logging` module just adds:

# * timestamps ⏰
# * log levels
# * file output
# * formatting
# * filtering

# …but the **core idea is identical**.

# ---

# ## 5️⃣ Mental model

# * `logger` = a polite `print`
# * `.info()` = “this is normal status”
# * `.warning()` = “pay attention”
# * `.error()` = “something failed”

# ---

# ## One-sentence takeaway 🧠

# > `logger.info` is just a structured, grown-up `print()`.

# If you want next:

# * I can build a *tiny* fake logging system step by step
# * Or explain when to use `info` vs `debug`
# * Or show how logging helps debug fallback logic like your router


The analogy (keep this forever) 🏗️
Building a house
LangChainRAG 🏠
You hire a general contractor.
You say:
“Build me a house at this address.”
They:
hire electricians
pour concrete
wire everything
hand you the keys
You don’t see the steps.
VectorRAG 🔨
You are the builder.
You:
pour the foundation
wire the walls
install plumbing
inspect everything
More work — but full control.


# What LangChain adds (that you don’t see yet)
LangChain quietly gives you:
document loaders
chunkers
retrievers
chains
memory helpers
Your VectorRAG is:
cleaner
lighter
easier to reason about
better for learning
One-sentence takeaway 🧠
LangChainRAG and VectorRAG both talk to the same kind of vector database — LangChain just hides the plumbing, while VectorRAG shows you every pipe.